# 04 - Embeddings & Semantic Search

**Learning objective:** Provision Managed Inference for embeddings and run semantic search over drug label chunks.

We will:
1. Live-provision Managed Inference (BGE Multilingual Gemma2) via `tofu apply` (~15-20 min)
2. Batch-embed all chunks
3. Run canonical similarity queries

> **The ~15-20 minute Managed Inference boot time is a good time for the instructor's ReAct theory lecture.**

In [ ]:
import os
import subprocess
import json
import sys
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, "..")

project_suffix = os.environ["PROJECT_SUFFIX"]
iac_dir = "../iac_snippets/managed_inference"

# Write tfvars
tfvars = f"""access_key      = "{os.environ["SCW_ACCESS_KEY"]}"
secret_key      = "{os.environ["SCW_SECRET_KEY"]}"
organization_id = "{os.environ["SCW_DEFAULT_ORGANIZATION_ID"]}"
project_id      = "{os.environ["SCW_DEFAULT_PROJECT_ID"]}"
project_suffix      = "{project_suffix}"
"""
with open(f"{iac_dir}/terraform.tfvars", "w") as f:
    f.write(tfvars)

print("Written terraform.tfvars. Starting Managed Inference provisioning...")
print("This takes ~15-20 minutes. Good time for the ReAct theory lecture!")

In [ ]:
# Initialize and apply (~15-20 minutes)
subprocess.run(["tofu", "init"], cwd=iac_dir, check=True)
result = subprocess.run(
    ["tofu", "apply", "-auto-approve"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

In [ ]:
# Get the endpoint URL
result = subprocess.run(
    ["tofu", "output", "-raw", "endpoint_url"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
endpoint_url = result.stdout.strip()
print(f"Embedding endpoint: {endpoint_url}")

In [ ]:
# Create embeddings client
from openai import OpenAI
from src.embeddings import EmbeddingsClient

embedding_openai = OpenAI(
    base_url=endpoint_url,
    api_key=os.environ["SCW_SECRET_KEY"],
)

embeddings = EmbeddingsClient(
    client=embedding_openai,
    model="baai/bge-multilingual-gemma2:fp16",
)

# Quick test
test_emb = embeddings.embed("test")
print(f"Embedding dimension: {len(test_emb)}")

In [ ]:
# Reconnect to database
import psycopg

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)
print("Connected to database.")

In [ ]:
# Batch-embed all chunks and update the database
with conn.cursor() as cur:
    cur.execute("SELECT id, text FROM chunks WHERE embedding IS NULL")
    rows = cur.fetchall()

print(f"Chunks to embed: {len(rows)}")

# Process in batches
batch_size = 32
for i in range(0, len(rows), batch_size):
    batch = rows[i : i + batch_size]
    texts = [row[1][:2000] for row in batch]  # Truncate very long texts
    embs = embeddings.embed_batch(texts)

    with conn.cursor() as cur:
        for (chunk_id, _), emb in zip(batch, embs):
            cur.execute(
                "UPDATE chunks SET embedding = %s WHERE id = %s",
                (emb, chunk_id),
            )
    conn.commit()

    if (i // batch_size) % 5 == 0:
        print(f"  Embedded {min(i + batch_size, len(rows))}/{len(rows)} chunks")

print("All chunks embedded!")

In [ ]:
# Canonical query 1: "NSAIDs gastrointestinal bleeding"
from src.rag import similarity_search

query_emb = embeddings.embed("NSAIDs gastrointestinal bleeding")
results = similarity_search(conn, query_emb, k=5)

print("Query: 'NSAIDs gastrointestinal bleeding'")
print("---")
for r in results:
    print(f"  Drug: {r['drug_name']} | Section: {r['section_type']} | Distance: {r['distance']:.3f}")
    print(f"  Text: {r['text'][:150]}...")
    print()

In [ ]:
# Canonical query 2: "third trimester NSAID avoidance"
query_emb = embeddings.embed("third trimester NSAID avoidance pregnancy")
results = similarity_search(conn, query_emb, k=5)

print("Query: 'third trimester NSAID avoidance'")
print("---")
for r in results:
    print(f"  Drug: {r['drug_name']} | Section: {r['section_type']} | Distance: {r['distance']:.3f}")
    print(f"  Text: {r['text'][:150]}...")
    print()

## You should now have

- [x] A running Managed Inference deployment (BGE Multilingual Gemma2)
- [x] All chunks embedded with 3584-dim vectors
- [x] Semantic search returning relevant drug label sections
- [x] Verified that NSAID/bleeding queries surface ibuprofen/aspirin chunks

**Next:** `05_tools.ipynb`